In [2]:
import pandas as pd
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Acs1112@mysql",
    database="behaviorpulse"
)

df = pd.read_sql("SELECT * FROM activities", conn)
df.head()


C:\Users\chira\AppData\Local\Temp\ipykernel_17344\3571463591.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM activities", conn)


,record_id,user_id,activity_date,day_of_week,activity_type,planned_time_hrs,actual_time_hrs,completion_status,interruption_count,energy_level,mood,invalid_time_flag,energy_missing_flag,duplicate_flag,efficiency,burnout_risk,high_interruption_flag
0,1,101,2025-11-01,mon,study,3.0,2.5,yes,2,4,high,0,0,0,0.833333,0,0
1,2,101,2025-11-02,tue,study,2.0,3.0,yes,5,3,neutral,0,0,0,1.500000,0,1
2,3,101,2025-11-03,wed,study,3.0,1.5,no,7,3,low,0,1,0,0.500000,0,1
3,4,101,2025-11-04,thu,work,4.0,4.5,yes,1,5,high,0,0,0,1.125000,0,0
4,5,101,2025-11-05,fri,rest,1.0,0.5,no,3,2,low,0,0,0,0.500000,0,0


In [3]:
df['activity_type'] = df['activity_type'].astype('category').cat.codes
df['day_of_week'] = df['day_of_week'].astype('category').cat.codes


In [4]:
features = [
    'planned_time_hrs',
    'actual_time_hrs',
    'interruption_count',
    'energy_level',
    'completion_status',
    'efficiency'
]

X = df[features]
y = df['burnout_risk']


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)
print(classification_report(y_test, preds))


ValueError: could not convert string to float: 'yes'

In [6]:
df = pd.read_sql("SELECT * FROM activities", conn)


C:\Users\chira\AppData\Local\Temp\ipykernel_17344\2040902322.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM activities", conn)


In [8]:
df['completion_status'] = df['completion_status'].map({'yes': 1, 'no': 0})


In [9]:
df['activity_type'] = df['activity_type'].astype('category').cat.codes


In [10]:
df['day_of_week'] = df['day_of_week'].astype('category').cat.codes


In [11]:
df.dtypes


record_id                   int64
user_id                     int64
activity_date              object
day_of_week                  int8
activity_type                int8
planned_time_hrs          float64
actual_time_hrs           float64
completion_status           int64
interruption_count          int64
energy_level                int64
mood                       object
invalid_time_flag           int64
energy_missing_flag         int64
duplicate_flag              int64
efficiency                float64
burnout_risk                int64
high_interruption_flag      int64
dtype: object

In [12]:
df['activity_date'] = pd.to_datetime(df['activity_date'])

df['day'] = df['activity_date'].dt.day
df['month'] = df['activity_date'].dt.month
df['weekday'] = df['activity_date'].dt.weekday

df = df.drop(columns=['activity_date'])


In [13]:
df['mood'] = df['mood'].astype('category').cat.codes


In [14]:
df.select_dtypes(include='object').columns


Index([], dtype='object')

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

features = [
    'planned_time_hrs',
    'actual_time_hrs',
    'interruption_count',
    'energy_level',
    'completion_status',
    'efficiency'
]

X = df[features]
y = df['burnout_risk']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)
print(classification_report(y_test, preds))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00        24

    accuracy                           1.00        24
   macro avg       1.00      1.00      1.00        24
weighted avg       1.00      1.00      1.00        24



In [16]:
import pandas as pd

importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values(by='importance', ascending=False)

importance


,feature,importance
0,planned_time_hrs,0.0
1,actual_time_hrs,0.0
2,interruption_count,0.0
3,energy_level,0.0
4,completion_status,0.0
5,efficiency,0.0


In [17]:
features_no_leak = [
    'planned_time_hrs',
    'actual_time_hrs',
    'interruption_count',
    'energy_level',
    'completion_status'
]

X = df[features_no_leak]
y = df['burnout_risk']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model2 = RandomForestClassifier(random_state=42)
model2.fit(X_train, y_train)

preds2 = model2.predict(X_test)
print(classification_report(y_test, preds2))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00        24

    accuracy                           1.00        24
   macro avg       1.00      1.00      1.00        24
weighted avg       1.00      1.00      1.00        24



In [18]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model2, X, y, cv=5)
print("CV scores:", scores)
print("Mean CV accuracy:", scores.mean())


CV scores: [1. 1. 1. 1. 1.]
Mean CV accuracy: 1.0


In [19]:
sample = pd.DataFrame([{
    'planned_time_hrs': 4,
    'actual_time_hrs': 6,
    'interruption_count': 8,
    'energy_level': 2,
    'completion_status': 0
}])

model2.predict(sample)


array([0])

In [20]:
import joblib

joblib.dump(model2, "burnout_model.pkl")


['burnout_model.pkl']

In [21]:
model = joblib.load("burnout_model.pkl")
